# 08-2 Choropleth Maps: Legionnaires' Disease by County/City in Taiwan (2003–present)

Inside the nursing home we used heatmaps and spot maps; to move up to the community level, we need a **choropleth (a shaded/graduated-color map)**.

This notebook uses two real government open datasets:
1. **National Land Surveying and Mapping Center (NLSC)** — special municipality, county, and city boundary SHP (TWD97 EPSG:3824)
2. **Taiwan CDC** — Legionnaires' disease statistics by area, age, and sex, from 2003 onward

Workflow: **download data → read the SHP → read the CDC CSV → normalize 台/臺 → ID matching → static map → year-by-year animation**

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys, os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e . pillow

In [ ]:
# --- Packages and font setup ---
import pathlib, urllib.request, zipfile, warnings, ssl
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.animation import FuncAnimation
from IPython.display import Image as IPyImage

for _fd in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _fd.exists():
        for _fp in sorted(_fd.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name):
                try: fm.fontManager.addfont(str(_fp))
                except Exception: pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS", "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 120
warnings.filterwarnings("ignore")
print("Packages loaded")

In [ ]:
# --- Step 1: Download government open data ---
DATA_DIR    = pathlib.Path("data/external")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# NLSC county/city boundaries (TWD97 EPSG:3824)
SHP_ZIP_URL = (
    "https://maps.nlsc.gov.tw/download/"
    "%E7%B8%A3%E5%B8%82%E7%95%8C%E7%B7%9A"
    "(TWD97%E7%B6%93%E7%B7%AF%E5%BA%A6).zip"
)
# CDC Legionnaires' disease statistics by area, age, and sex (by onset month, 2003–)
CDC_CSV_URL = "https://od.cdc.gov.tw/eic/Age_County_Gender_4828.csv"

SHP_ZIP = DATA_DIR / "tw_county_boundary.zip"
SHP_DIR = DATA_DIR / "tw_county_shp"
CDC_CSV = DATA_DIR / "tw_legionella_4828.csv"


def _ssl_ctx(insecure: bool = False) -> ssl.SSLContext:
    """Build an SSL context; when insecure=True, disable certificate verification (for government sites with non-compliant certs)."""
    if insecure:
        ctx = ssl.create_default_context()
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
        return ctx
    return ssl.create_default_context()


def _download(url: str, dest: pathlib.Path, timeout: int = 45) -> bool:
    """Download url → dest; skip if it already exists; automatically retry with verification disabled on SSL failure."""
    if dest.exists():
        print(f"✓ Already cached: {dest.name}")
        return True
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    # First pass: strict verification; second pass: verification disabled (for Taiwan government certs missing the SKID)
    for insecure in (False, True):
        try:
            with urllib.request.urlopen(req, timeout=timeout, context=_ssl_ctx(insecure)) as resp:
                dest.write_bytes(resp.read())
            tag = " (⚠ SSL verification disabled)" if insecure else ""
            print(f"✓ Downloaded: {dest.name} ({dest.stat().st_size // 1024} KB){tag}")
            return True
        except urllib.error.URLError as e:
            msg = str(e)
            if not insecure and ("CERTIFICATE_VERIFY" in msg or "SSL" in msg):
                print(f"  SSL verification failed → retrying with verification disabled ({dest.name})")
                continue
            print(f"✗ Download failed ({dest.name}): {type(e).__name__}: {e}")
            return False
        except Exception as e:
            print(f"✗ Download failed ({dest.name}): {type(e).__name__}: {e}")
            return False
    return False


shp_ok = _download(SHP_ZIP_URL, SHP_ZIP)
cdc_ok = _download(CDC_CSV_URL, CDC_CSV)

_DEMO = not (shp_ok and cdc_ok)
if _DEMO:
    print("\n⚠️  Some data could not be downloaded; synthetic demo data will be used from here on")
    print("    Re-run once you have a network connection to get the real data")
else:
    print("\n✅ Data download complete")

In [ ]:
# --- Step 2: Read the county/city boundary SHP ---
# The NLSC SHP uses EPSG:3824 (TWD97 lat/long), which converts directly to WGS84 (EPSG:4326)

# Whether the GEOMETRY is synthetic. Tracked separately from _DEMO: a failed
# stats download sets _DEMO=True even when the real map geometry did load, so
# the view window must key on the geometry source, not on _DEMO.
_DEMO_GEOM = False

if not _DEMO:
    # Unzip the SHP zip
    if not SHP_DIR.exists():
        with zipfile.ZipFile(SHP_ZIP) as zf:
            zf.extractall(SHP_DIR)
            print(f"Extracted to {SHP_DIR}/")

    shp_files = sorted(SHP_DIR.rglob("*.shp"))
    if not shp_files:
        print("⚠️  No .shp found inside the ZIP; falling back to demo data"); _DEMO = True
    else:
        print(f"SHP files: {[f.name for f in shp_files]}")
        gdf = gpd.read_file(shp_files[0])
        # TWD97 → WGS84 (so matplotlib can plot it directly)
        gdf = gdf.to_crs(epsg=4326)
        print(f"\nNumber of counties/cities: {len(gdf)}")
        print(f"Columns: {list(gdf.columns)}")

        # Auto-detect the county-name column (find a string column containing "縣" or "市")
        county_col = None
        for col in gdf.select_dtypes(include="object").columns:
            vals = gdf[col].dropna().head(10).tolist()
            if any(("縣" in str(v) or "市" in str(v)) for v in vals):
                county_col = col
                break
        if county_col is None:
            county_col = gdf.select_dtypes(include="object").columns[0]
        print(f"\nCounty-name column: {county_col!r}")
        print(gdf[[county_col]].sort_values(county_col).to_string(index=False))

if _DEMO:
    # When the real boundaries can't be downloaded, fall back to a "tile-grid
    # map (tilegram)": each county = one equal-sized, non-overlapping square,
    # placed to loosely match Taiwan's geography (north up, south down, outlying
    # islands to the west). This always renders offline and — unlike squares
    # stacked on real centroids — never hides one county's color behind another.
    from shapely.geometry import Polygon
    TILE_LAYOUT = {
        "連江縣": (1, 10), "基隆市": (4, 9), "臺北市": (3, 9), "新北市": (3, 8),
        "桃園市": (2, 8),  "宜蘭縣": (4, 8), "新竹市": (2, 7), "新竹縣": (3, 7),
        "金門縣": (0, 6),  "苗栗縣": (2, 6), "臺中市": (3, 6), "花蓮縣": (4, 6),
        "彰化縣": (2, 5),  "南投縣": (3, 5), "澎湖縣": (0, 4), "雲林縣": (2, 4),
        "嘉義市": (2, 3),  "嘉義縣": (3, 3), "臺南市": (2, 2), "高雄市": (2, 1),
        "臺東縣": (3, 1),  "屏東縣": (2, 0),
    }
    DEMO_NAMES = list(TILE_LAYOUT)
    polys = [Polygon([(x, y), (x + 0.92, y), (x + 0.92, y + 0.92), (x, y + 0.92)])
             for x, y in TILE_LAYOUT.values()]
    gdf = gpd.GeoDataFrame({"COUNTYNAME": DEMO_NAMES, "geometry": polys}, crs="EPSG:4326")
    county_col = "COUNTYNAME"
    _DEMO_GEOM = True
    print(f"Using {len(gdf)} county tiles (tilegram demo — not real geography)")

In [ ]:
# --- Step 3: Read the CDC Legionnaires' disease surveillance data ---
# Note: normalize 台/臺 + convert old county names (pre-2010) → current names

# 台/臺 lookup table (the official form is "臺"; the NLSC SHP also uses "臺")
# Also handles the 2010 county mergers: 台北縣→新北市, 台中縣/市→臺中市, etc.
TAI_NORMALIZE = {
    # 台 → 臺 (4 counties/cities have this difference)
    "台北市": "臺北市",  "台中市": "臺中市",
    "台南市": "臺南市",  "台東縣": "臺東縣",
    # 2010 reorganization: old county/city name → current name
    "臺北縣": "新北市",  "台北縣": "新北市",
    "臺中縣": "臺中市",  "台中縣": "臺中市",
    "臺南縣": "臺南市",  "台南縣": "臺南市",
    "高雄縣": "高雄市",  "桃園縣": "桃園市",
}

def normalize_county(name: str) -> str:
    """Normalize 台/臺 + unify pre-2010 names to their current form."""
    return TAI_NORMALIZE.get(str(name).strip(), str(name).strip())

if not _DEMO:
    # Read the CSV (auto-detect encoding)
    for enc in ("utf-8-sig", "utf-8", "cp950"):
        try:
            raw = pd.read_csv(CDC_CSV, encoding=enc)
            break
        except Exception:
            continue

    raw.columns = [c.strip() for c in raw.columns]
    print("=== CDC data columns ===")
    print(raw.columns.tolist())
    print(raw.head(3).to_string())

    # Auto-detect column names (to handle different versions of the CDC OD format)
    def _find(candidates):
        for c in candidates:
            if c in raw.columns: return c
        return None

    year_col   = _find(["發病年份","年份","Year","year"])
    county_col_cdc = _find(["縣市","County","county","行政區"])
    cases_col  = _find(["確定病例數","病例數","Cases","cases","個案數"])
    print(f"\nDetected: year={year_col}, county={county_col_cdc}, cases={cases_col}")

    if not all([year_col, county_col_cdc, cases_col]):
        print("⚠️  Column detection failed; falling back to demo data"); _DEMO = True
    else:
        df = raw[[year_col, county_col_cdc, cases_col]].copy()
        df.columns = ["year", "county", "cases"]
        df["cases"]  = pd.to_numeric(df["cases"], errors="coerce").fillna(0).astype(int)
        df["county"] = df["county"].apply(normalize_county)
        # Keep only counties that have a map counterpart
        valid = set(gdf[county_col].apply(normalize_county))
        df = df[df["county"].isin(valid)].copy()
        print(f"\n{len(df)} rows, years: {df['year'].min()}–{df['year'].max()}")
        print(f"Number of counties: {df['county'].nunique()}")

if _DEMO:
    rng1 = np.random.default_rng(42)
    demo_counties = gdf[county_col].tolist()
    north = {"臺北市","新北市","桃園市","基隆市","新竹市","新竹縣"}
    records = []
    for yr in range(2003, 2025):
        for cn in demo_counties:
            base = 3 if cn in north else 1
            records.append({
                "year": yr, "county": cn,
                "cases": max(0, int(rng1.poisson(base * (1 + (yr-2003)*0.04))))
            })
    df = pd.DataFrame(records)
    print(f"Synthetic demo data: {len(df)} rows, {df['year'].min()}–{df['year'].max()}")

In [ ]:
# --- Step 4: ID matching debug (checking county-name consistency) ---
# This is the most important step before making a choropleth: confirm the county names in the SHP and CSV match exactly

shp_counties  = set(gdf[county_col].apply(normalize_county))
data_counties = set(df["county"].unique())

print("=== County ID matching results ===")
only_in_shp  = sorted(shp_counties - data_counties)
only_in_data = sorted(data_counties - shp_counties)
matched      = sorted(shp_counties & data_counties)

if only_in_shp:
    print(f"\n⚠️  Only in SHP (will appear blank on the map): {only_in_shp}")
else:
    print("\n✅ Every county in the SHP has a counterpart in the CDC data")

if only_in_data:
    print(f"\n⚠️  Only in CDC data (won't be shown on the map): {only_in_data}")
    print("    (May be non-county codes like 'unknown' or 'out of area'; already filtered in the previous step)")
else:
    print("✅ Every county in the CDC data has a counterpart in the SHP")

print(f"\n✅ Successfully matched {len(matched)} counties/cities")

In [ ]:
# --- Step 5: Aggregate by year × county and compute the incidence rate per 100,000 ---
# Legionnaires' disease has a low notification rate and small absolute case counts; the incidence rate (per 100k) compares counties more fairly

# Approximate county populations (2023 estimates, in people)
COUNTY_POP = {
    "臺北市": 2_530_000, "新北市": 4_036_000, "桃園市": 2_307_000,
    "臺中市": 2_854_000, "臺南市": 1_876_000, "高雄市": 2_761_000,
    "基隆市":   370_000, "新竹市":   454_000, "新竹縣":   574_000,
    "苗栗縣":   543_000, "彰化縣": 1_284_000, "南投縣":   487_000,
    "雲林縣":   673_000, "嘉義市":   268_000, "嘉義縣":   504_000,
    "屏東縣":   822_000, "宜蘭縣":   461_000, "花蓮縣":   325_000,
    "臺東縣":   223_000, "澎湖縣":   107_000, "金門縣":   143_000,
    "連江縣":    14_000,
}

annual = df.groupby(["year","county"])["cases"].sum().reset_index()
annual["population"]   = annual["county"].map(COUNTY_POP).fillna(500_000)
annual["rate_per_100k"] = (annual["cases"] / annual["population"] * 100_000).round(3)

print(f"Year range: {annual['year'].min()}–{annual['year'].max()}")
print(f"Total notified cases: {annual['cases'].sum()}")
print("\nTop 5 counties by cumulative incidence rate (per 100,000):")
top5 = (annual.groupby("county")
              .agg(total_cases=("cases","sum"), avg_rate=("rate_per_100k","mean"))
              .nlargest(5,"avg_rate"))
print(top5.round(3).to_string())

In [ ]:
# --- Step 6: Static choropleth (latest year) ---
# View extent: real geometry -> fixed Taiwan main-island clip (drops distant
#   islands like Dongsha / Taiping); synthetic geometry (tilegram) -> derived
#   from the data extent so it is guaranteed to render.
TW_CLIP = (119.323286, 122.128611, 21.739091, 25.621716)  # (lon_min, lon_max, lat_min, lat_max)

def view_window(geo, fallback=TW_CLIP, pad=0.04):
    """Derive a safe view window from the geometry extent; fall back if the
    bounds are NaN or degenerate (empty / collinear)."""
    valid = geo.geometry.dropna()
    if len(valid) == 0:
        return fallback
    minx, miny, maxx, maxy = valid.total_bounds
    if not np.all(np.isfinite([minx, miny, maxx, maxy])) or minx >= maxx or miny >= maxy:
        return fallback
    mx, my = (maxx - minx) * pad, (maxy - miny) * pad
    return (minx - mx, maxx + mx, miny - my, maxy + my)

latest_year = int(annual["year"].max())
latest = annual[annual["year"] == latest_year].copy()

# Merge the map with the statistics
gdf_plot = gdf.copy()
gdf_plot["county_norm"] = gdf_plot[county_col].apply(normalize_county)
gdf_merged = gdf_plot.merge(
    latest[["county", "rate_per_100k", "cases"]],
    left_on="county_norm", right_on="county", how="left"
)
gdf_merged["rate_per_100k"] = gdf_merged["rate_per_100k"].fillna(0)

# Choose the window by GEOMETRY source (not _DEMO): real geometry keeps the
# fixed clip even when paired with demo case counts.
LON_MIN, LON_MAX, LAT_MIN, LAT_MAX = view_window(gdf_merged) if _DEMO_GEOM else TW_CLIP

fig, ax = plt.subplots(figsize=(7, 9), dpi=150)
fig.patch.set_facecolor("#FAF8F3")
ax.set_facecolor("#FAF8F3")

gdf_merged.plot(
    column="rate_per_100k",
    ax=ax,
    cmap="Reds",
    legend=True,
    legend_kwds={
        "label": "Incidence rate (per 100,000)",
        "orientation": "horizontal",
        "shrink": 0.6,
        "pad": 0.01,
    },
    edgecolor="white",
    linewidth=0.5,
    missing_kwds={"color": "#E8E5DF", "label": "No data"},
)
# Tilegram demo mode: label each tile with its county name (skipped for the
# real map to avoid overprinting).
if _DEMO_GEOM:
    import matplotlib.patheffects as _pe
    _halo = [_pe.withStroke(linewidth=1.8, foreground="white")]
    for _, _r in gdf_merged.iterrows():
        _c = _r.geometry.centroid
        ax.annotate(_r[county_col], (_c.x, _c.y), ha="center", va="center",
                    fontsize=6, color="#1A1A1A", path_effects=_halo)
# Restrict the view (static map is drawn once, so no ax.clear concerns here)
ax.set_xlim(LON_MIN, LON_MAX)
ax.set_ylim(LAT_MIN, LAT_MAX)
ax.set_aspect("equal")
_src_note = (
    "Taiwan CDC open data" if not _DEMO
    else "demo data, tilegram schematic map" if _DEMO_GEOM
    else "demo data, real county borders"
)
ax.set_title(
    f"{latest_year} Legionnaires' disease incidence rate in Taiwan (per 100,000)\n"
    f"(Source: {_src_note})",
    fontsize=12, pad=10
)
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"\n→ {latest_year}: {latest['cases'].sum()} cases notified in total")
print("→ Top 3 counties:")
for _, row in latest.nlargest(3, "rate_per_100k").iterrows():
    print(f"   {row['county']}: {row['cases']} cases ({row['rate_per_100k']:.3f}/100k)")

In [ ]:
# --- Step 7: Year-by-year animated choropleth (FuncAnimation → GIF) ---
# Show how each county's incidence rate has changed over time since 2003
# The view extent carries over from Step 6 (LON_MIN..LON_MAX, LAT_MIN..LAT_MAX,
# already chosen by geometry source)

years = sorted(annual["year"].unique())
# Use the 95th percentile as the color-axis upper bound (so a few high values don't compress the overall colors)
vmax  = annual["rate_per_100k"].quantile(0.95)
vmax  = max(vmax, 0.1)  # avoid vmax = 0

gdf_base = gdf.copy()
gdf_base["county_norm"] = gdf_base[county_col].apply(normalize_county)

fig, ax = plt.subplots(figsize=(5, 7), dpi=100)
fig.patch.set_facecolor("#FAF8F3")

def _update(year):
    ax.clear()
    ax.set_facecolor("#FAF8F3")
    yr_data = annual[annual["year"] == year][["county","rate_per_100k"]]
    merged  = gdf_base.merge(yr_data, left_on="county_norm", right_on="county", how="left")
    merged["rate_per_100k"] = merged["rate_per_100k"].fillna(0)
    merged.plot(
        column="rate_per_100k", ax=ax,
        cmap="Reds", vmin=0, vmax=vmax,
        edgecolor="white", linewidth=0.5,
        legend=False,
    )
    # xlim/ylim must be reset every frame (ax.clear() wipes them)
    ax.set_xlim(LON_MIN, LON_MAX)
    ax.set_ylim(LAT_MIN, LAT_MAX)
    ax.set_aspect("equal")
    ax.set_title(f"Legionnaires' disease incidence rate in Taiwan\n{year} (per 100,000)", fontsize=11)
    ax.axis("off")

anim = FuncAnimation(fig, _update, frames=years, interval=800, repeat=True)

# Save as a GIF (using the pillow writer, dpi=100 is fine for web embedding)
gif_path = DATA_DIR / "tw_legionella_anim.gif"
try:
    anim.save(str(gif_path), writer="pillow", fps=1, dpi=100)
    plt.close()
    print(f"✓ Animation saved to: {gif_path}")
    print(f"  {len(years)} frames, covering {years[0]}–{years[-1]}")
    print(f"  Resolution: 100 dpi ({gif_path.stat().st_size // 1024} KB)")
    display(IPyImage(filename=str(gif_path)))
except Exception as e:
    plt.close()
    print(f"⚠️  Could not save GIF: {e}")

## Step 8: Switch to a real map of Taiwan (bundled GeoJSON)

The tilegram above is a schematic grid for the offline fallback. Here we use the project's **bundled real Taiwan county boundaries** `data/geojson/county_smooth_inset.geojson`: the borders are smoothed, and the outlying islands (Matsu, Kinmen, Penghu) are scaled and laid out as **insets** to the west of the main island — a genuine map of Taiwan that needs no network.

- **Bundled in the repo**: works offline, on Colab, and in the book build — it always renders (exactly what we wanted at the start but couldn't download live from government sites).
- **Case data reuses** the `annual` table from Step 5 (real or demo), joined on the normalized `COUNTYNAME`.

In [ ]:
# --- Step 8: Real Taiwan county choropleth (bundled GeoJSON, works offline) ---
# The tilegram is an offline schematic; here we use the repo's bundled real
# county boundaries (smoothed, with the outlying islands laid out as insets).
import matplotlib.patheffects as pe

def _locate(rel):
    """Walk up from cwd to find a file, robust to the book-build / standalone /
    Colab working directories."""
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (base / rel).exists():
            return base / rel
    return pathlib.Path(rel) if pathlib.Path(rel).exists() else None

geojson_path = _locate("data/geojson/county_smooth_inset.geojson")
if geojson_path is None:
    print("⚠️  Bundled GeoJSON not found (data/geojson/county_smooth_inset.geojson); skipping the real map")
else:
    real = gpd.read_file(geojson_path)
    real["county_norm"] = real["COUNTYNAME"].apply(normalize_county)

    # Reuse the annual table from Step 5; take the latest year's incidence rate
    _yr = int(annual["year"].max())
    _rate = annual[annual["year"] == _yr][["county", "rate_per_100k", "cases"]]
    real = real.merge(_rate, left_on="county_norm", right_on="county", how="left")
    real["rate_per_100k"] = real["rate_per_100k"].fillna(0)

    fig, ax = plt.subplots(figsize=(7, 8), dpi=150)
    fig.patch.set_facecolor("#FAF8F3")
    ax.set_facecolor("#FAF8F3")
    real.plot(
        column="rate_per_100k", ax=ax, cmap="Reds",
        edgecolor="#8A8A8A", linewidth=0.4, legend=True,
        legend_kwds={"label": "Incidence rate (per 100,000)", "orientation": "horizontal",
                     "shrink": 0.6, "pad": 0.01},
        missing_kwds={"color": "#E8E5DF", "label": "No data"},
    )
    # Label each county in English (white halo keeps labels legible on dark tiles)
    _halo = [pe.withStroke(linewidth=1.8, foreground="white")]
    for _, _row in real.iterrows():
        _pt = _row.geometry.representative_point()
        ax.annotate(_row["COUNTYENG"], (_pt.x, _pt.y), ha="center", va="center",
                    fontsize=5.0, color="#1A1A1A", path_effects=_halo)
    # The inset layout already moved the islands west of the main island, so
    # frame the view from the data extent.
    minx, miny, maxx, maxy = real.total_bounds
    ax.set_xlim(minx - 0.1, maxx + 0.1)
    ax.set_ylim(miny - 0.1, maxy + 0.1)
    ax.set_aspect("equal")
    ax.set_title(
        f"{_yr} Legionnaires' disease incidence rate in Taiwan (per 100,000)\n"
        f"(Real county map; outlying islands shown as insets)",
        fontsize=12, pad=10,
    )
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    print(f"→ Using the bundled real map: {geojson_path}")
    print(f"→ {real['COUNTYENG'].nunique()} counties; highest {_yr} incidence rates:")
    for _, _r in real.nlargest(3, "rate_per_100k").iterrows():
        print(f"   {_r['COUNTYENG']}: {_r['rate_per_100k']:.3f}/100k")

## Summary

| Step | Skill learned |
|---|---|
| Download government SHP + CSV | `urllib.request` with a User-Agent to get past 403s |
| Read the SHP | `geopandas.read_file()` + `to_crs(epsg=4326)` |
| Auto-detect columns | String search to find the county-name column |
| Normalize 台/臺 | `str.replace` + a lookup table for pre/post-reorganization names |
| ID matching debug | `set.difference()` to find mismatched counties |
| Static choropleth | `gdf.merge()` + `gdf.plot(column=..., cmap=...)` |
| Animated choropleth | `FuncAnimation` → `anim.save(..., writer="pillow")` |
| Real county map | Bundled GeoJSON (county borders + island insets) + `gpd.merge` |

### Key concepts

- **台/臺**: government documents, the CDC, and the NLSC officially use "臺"; online articles often use "台". You must normalize before the JOIN.
- **2010 reorganization**: Taipei County → New Taipei City; Taichung County/City → Taichung City; data from 2003 uses the old names, so a lookup table is needed to convert them.
- **Incidence rate per 100,000 vs. absolute case counts**: a populous county having more cases doesn't mean its risk is higher; you must always standardize by population.
- **Synthetic demo data**: if there's no network connection, the notebook automatically switches to synthetic data — the concept is the same, but the numbers don't reflect reality.
- **Bundled real map**: committing a county GeoJSON to the repo renders the real Taiwan map offline too, without relying on live downloads from government sites.